### Load the test runner context

**Purpose:** Prepare the notebook runtime for the project test command.

**Inputs:** Demo2 Olist source and test directories.

**Outputs:** A controlled test-runner environment.

**Why it matters:** This notebook is the Job entry point for automated checks, not a replacement for individual test modules.

In [0]:
 %pip install -q pytest

### Load project test configuration

**Purpose:** Establish the paths and runtime settings used by the automated test entry point.

**Inputs:** Demo2 Olist source, unit, and integration test directories.

**Outputs:** Test configuration for the remaining cells.

**Why it matters:** The Job needs a deterministic test boundary before final validation.

### Prepare the test interpreter

**Purpose:** Restart or prepare Python after test dependencies are installed.

**Inputs:** The notebook environment and pytest dependency.

**Outputs:** A clean test execution context.

**Why it matters:** The Job should execute tests with the dependencies declared for this project.

In [0]:
# Purpose: run unit and integration tests without writing cache files
# into the Databricks Workspace filesystem.

from pathlib import Path
import os
import sys
import pytest

project_root = Path(
    "/Workspace/Users/parvinbadalov@softserve.academy/"
    "Databricks-Academy-Lakehouse/Demos/Demo2_Olist"
)

print("Project root:", project_root)

# Make the project importable.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Databricks /Workspace does not support Python cache directories.
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

result = pytest.main(
    [
        str(project_root / "tests/unit"),
        str(project_root / "tests/integration"),
        "-v",

        # Prevent creation of __pycache__ for assertion rewriting.
        "--assert=plain",

        # Prevent creation of .pytest_cache.
        "-p",
        "no:cacheprovider",
    ]
)

if result != pytest.ExitCode.OK:
    raise RuntimeError(f"Tests failed with pytest exit code: {result}")

print("SUCCESS: all unit and integration tests passed")